In [1]:
import os
os.chdir('../')
os.getcwd()

'/home/minh_khai/skin_disease/skin-disease'

## Load model

In [2]:
from abc import ABC, abstractmethod
from pathlib import Path

class BaseVisionModel(ABC):
    @abstractmethod
    def predict(self, imgs): ...
    
    @abstractmethod
    def save(self, path: Path) -> None: ...
    
    @abstractmethod
    def load(self, path: Path) -> None: ...
    
    @abstractmethod
    def save_onnx(self, path: Path, sample_input) -> None: ...

In [3]:
import torch.nn as nn
from torchvision.models import efficientnet_b0, resnet50

def build_efficientnet(num_classes: int, freeze_backbone: bool = True) -> nn.Module:
    model = efficientnet_b0(weights="IMAGENET1K_V1")
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    if freeze_backbone:
        for p in model.features.parameters():
            p.requires_grad = False
    return model

def build_resnet(num_classes: int, freeze_backbone: bool = True) -> nn.Module:
    model = resnet50(weights="IMAGENET1K_V1")
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    if freeze_backbone:
        for name, p in model.named_parameters():
            if not name.startswith("fc"):
                p.requires_grad = False
    return model

In [4]:
import torch
from pathlib import Path
import numpy as np
import onnxruntime as ort
import numpy as np
from pathlib import Path


def export_onnx(model, sample_input, output_path: Path, input_names=("input",), output_names=("output",), dynamic_axes=None):
    model.eval()
    torch.onnx.export(
        model,
        sample_input,
        str(output_path),
        input_names=list(input_names),
        output_names=list(output_names),
        dynamic_axes=dynamic_axes or {"input": {0: "batch_size"}, "output": {0: "batch_size"}},
        opset_version=17,
    )

class EfficientNetModel(BaseVisionModel):
    def __init__(self, num_classes: int, checkpoint_dir: Path = None):
        self.num_classes = num_classes
        self.checkpoint_dir = checkpoint_dir
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model  = build_efficientnet(num_classes, freeze_backbone=True).to(self.device)

    def predict(self, imgs):
        self.model.eval()
        with torch.no_grad():
            logits = self.model(imgs.to(self.device))
            probs = torch.softmax(logits, dim=1)
            confidence, pred_idx = torch.max(probs, dim=1)
        return pred_idx.item(), confidence.item()

    def save(self, path: Path) -> None:
        torch.save(self.model.state_dict(), path)

    def load(self, path: Path) -> None:
        checkpoint = torch.load(path, map_location=self.device)
        state = checkpoint["model_state"] if "model_state" in checkpoint else checkpoint
        self.model.load_state_dict(state)

    def save_onnx(self, path: Path, sample_input) -> None:
        export_onnx(self.model, sample_input, path)

2026-06-18 16:49:10.304046988 [W:onnxruntime:Default, device_discovery.cc:283 GetGpuDevices] Failed to detect devices under "/sys/class/drm/card0": device_discovery.cc:93 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"


In [5]:
from pathlib import Path

MODEL_REGISTRY = {
    "efficientnet": EfficientNetModel,
    # "resnet": ResNetModel,
}

class EvalModelFactory:
    @staticmethod
    def create(model_name: str, num_classes: int) -> BaseVisionModel:
        cls = MODEL_REGISTRY.get(model_name)
        if not cls:
            raise ValueError(f"No model registered for: '{model_name}'.")
        return cls(num_classes)

## Evaluation

In [6]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class EvaluationMetrics:
    model_name: str
    batch_size: int

@dataclass(frozen=True)
class EvaluationConfig:
    train_dir:  Path
    output_dir: Path
    data_dir:   Path
    metrics:    EvaluationMetrics

In [7]:
import os
from core.constants import *
from core import read_yaml, create_directories

class EvalConfigurationManager:
    def __init__(self, 
                 config_filepath=CONFIG_FILE_PATH, 
                 params_filepath=PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])
        
    def get_eval_config(self) -> EvaluationConfig:
        config = self.config.eval_config
        params = self.params.eval_params
        
        output_dir = Path(config.output_dir) / params.model_name
        create_directories([config.output_dir, output_dir])

        return EvaluationConfig(
            train_dir   = Path(config.train_dir),
            output_dir  = output_dir,
            data_dir    = Path(config.data_dir),
            metrics     = EvaluationMetrics(
                model_name  = params.model_name,
                batch_size  = params.batch_size,
            )
        )

In [8]:
import numpy as np
from torchvision import datasets
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

def get_eval_transforms():
    return transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])


def build_eval_loaders(eval_dir, batch_size: int, num_workers: int = 2):
    eval_ds = datasets.ImageFolder(eval_dir, transform=get_eval_transforms())
    eval_loader = DataLoader(eval_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    return eval_loader, eval_ds.classes

In [9]:
import torch
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

def compute_full_metrics(y_true, y_pred, class_names: list[str]) -> dict:
    report = classification_report(
        y_true, y_pred, target_names=class_names, output_dict=True, zero_division=0
    )
    cm = confusion_matrix(y_true, y_pred)
    return {
        "report": report,
        "confusion_matrix": cm.tolist(),
        "class_names": class_names,
    }
    
def collect_predictions(model, loader) -> tuple[np.ndarray, np.ndarray]:
    model.model.eval()
    y_true, y_pred = [], []

    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(model.device)
            logits = model.model(imgs)
            preds = logits.argmax(dim=1).cpu().numpy()

            y_pred.extend(preds)
            y_true.extend(labels.numpy())

    return np.array(y_true), np.array(y_pred)

In [10]:
import sys
from pathlib import Path
from core.logger import logger
from core.exception import CustomException

class Evaluation:
    def __init__(self, config: EvaluationConfig):
        self.config = config

    def evaluate(self):
        try:
            eval_loader, classes = build_eval_loaders(self.config.data_dir, self.config.metrics.batch_size)
            logger.info(f"Evaluation dataset: {len(eval_loader.dataset):,} Classes: {classes}")

            model = EvalModelFactory.create(self.config.metrics.model_name, len(classes))
            best_checkpoint = self._find_best_checkpoint()
            model.load(best_checkpoint)

            y_true, y_pred = collect_predictions(model, eval_loader)
            metrics = compute_full_metrics(y_true, y_pred, classes)

            logger.info(f"Eval accuracy: {metrics['report']['accuracy']:.4f}")
            return metrics

        except Exception as e:
            raise CustomException(e, sys)

    def _find_best_checkpoint(self) -> Path:
        ckpt_dir = self.config.train_dir / self.config.metrics.model_name
        checkpoints = list(ckpt_dir.glob("epoch*.pth"))
        if not checkpoints:
            raise FileNotFoundError(f"No checkpoint found in {ckpt_dir}")
        return max(checkpoints, key=lambda p: float(p.stem.split("_acc")[1]))

In [11]:
try:
    cfg_manager = EvalConfigurationManager()
    eval_cfg    = cfg_manager.get_eval_config()
    evaluation  = Evaluation(config=eval_cfg)
    evaluation.evaluate()
except Exception as e:
    raise CustomException(e, sys)

[2026-06-18 16:49:12,945: INFO: __init__: yaml file: config/config.yaml loaded successfully]
[2026-06-18 16:49:12,950: INFO: __init__: yaml file: params.yaml loaded successfully]
[2026-06-18 16:49:12,952: INFO: __init__: created directory at: artifacts]
[2026-06-18 16:49:12,954: INFO: __init__: created directory at: artifacts/evaluation]
[2026-06-18 16:49:12,956: INFO: __init__: created directory at: artifacts/evaluation/efficientnet]
[2026-06-18 16:49:12,968: INFO: 4062274419: Evaluation dataset: 4,073 Classes: ['1. Eczema 1677', '10. Warts Molluscum and other Viral Infections - 2103', '2. Melanoma 15.75k', '3. Atopic Dermatitis - 1.25k', '4. Basal Cell Carcinoma (BCC) 3323', '5. Melanocytic Nevi (NV) - 7970', '6. Benign Keratosis-like Lesions (BKL) 2624', '7. Psoriasis pictures Lichen Planus and related diseases - 2k', '8. Seborrheic Keratoses and other Benign Tumors - 1.8k', '9. Tinea Ringworm Candidiasis and other Fungal Infections - 1.7k']]
[2026-06-18 16:49:24,060: INFO: 40622744

## Evaluation with GRAD-CAM

Grad-CAM (Gradient-weighted Class Activation Mapping) là kỹ thuật giải thích cho mô hình CNN, giúp trực quan hóa vùng nào trên ảnh đầu vào ảnh hưởng nhiều nhất đến quyết định phân loại của model.

**How it works**

Khi CNN dự đoán "đây là Eczema", model không tự giải thích tại sao. Grad-CAM bóc lộ điều đó bằng cách:
1. Cho ảnh đi qua model như bình thường, lấy ra activation map (feature map) ở layer convolution cuối cùng — đây là nơi model đã học được các pattern không gian (vùng nào, hình dạng gì)
2. Tính gradient của output class (ví dụ "Eczema") so với activation map đó — gradient cao nghĩa là vùng đó ảnh hưởng mạnh đến quyết định cuối
3. Weighted sum activation map theo gradient → tạo ra 1 heatmap — vùng nóng (đỏ) là nơi model "chú ý" nhiều nhất để đưa ra dự đoán

**Why use?**

Nếu model dự đoán đúng "Melanoma" nhưng heatmap lại sáng ở góc ảnh, nền, hoặc vùng da khỏe mạnh — nghĩa là model đang học sai pattern (ví dụ học theo ánh sáng, góc chụp, hoặc artifact của dataset), không thực sự học đặc điểm bệnh lý. Dù accuracy cao, model này không tin được khi gặp ảnh thực tế khác phân bố data train.

In [12]:
import torch
import numpy as np
import seaborn as sns
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

class GradCAM:
    def __init__(self, model_wrapper):
        self.model = model_wrapper.model
        self.device = model_wrapper.device
        self.gradients = None
        self.activations = None
        
        target_layer = self.model.features[-1]
        target_layer.register_forward_hook(self._save_activation)
        target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, input, output):
        self.activations = output.detach()

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def generate(self, img_tensor: torch.Tensor, target_class: int = None) -> np.ndarray:
        self.model.eval()
        img_tensor = img_tensor.unsqueeze(0).to(self.device)
        img_tensor.requires_grad_()

        output = self.model(img_tensor)
        if target_class is None:
            target_class = output.argmax(dim=1).item()

        self.model.zero_grad()
        output[0, target_class].backward()

        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1).squeeze()
        cam = torch.relu(cam)
        cam = cam.cpu().numpy()
        cam = cv2.resize(cam, (img_tensor.shape[-1], img_tensor.shape[-2]))
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam

def save_confusion_matrix(cm, class_names: list[str], output_path: Path):
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title("Confusion Matrix")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(output_path, dpi=150)
    plt.close()

def save_gradcam_overlay(original_img: np.ndarray, cam: np.ndarray, output_path: Path, alpha: float = 0.5):
    heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    overlay = (alpha * heatmap + (1 - alpha) * original_img).astype(np.uint8)

    plt.figure(figsize=(6, 6))
    plt.imshow(overlay)
    plt.axis("off")
    plt.savefig(output_path, bbox_inches="tight", dpi=150)
    plt.close()

In [13]:
import sys, json
from pathlib import Path
from core.logger import logger
from core.exception import CustomException

class Evaluation:
    def __init__(self, config: EvaluationConfig):
        self.config = config

    def evaluate(self):
        try:
            eval_loader, classes = build_eval_loaders(self.config.data_dir, self.config.metrics.batch_size)
            logger.info(f"Evaluation dataset: {len(eval_loader.dataset):,} Classes: {classes}")

            model = EvalModelFactory.create(self.config.metrics.model_name, len(classes))
            best_checkpoint = self._find_best_checkpoint()
            model.load(best_checkpoint)

            y_true, y_pred = collect_predictions(model, eval_loader)
            metrics = compute_full_metrics(y_true, y_pred, classes)

            self._save_artifacts(metrics, classes)
            self._save_gradcam_samples(model, eval_loader, classes)
            
            logger.info(f"Eval accuracy: {metrics['report']['accuracy']:.4f}")
            return metrics

        except Exception as e:
            raise CustomException(e, sys)

    def _find_best_checkpoint(self) -> Path:
        ckpt_dir = self.config.train_dir / self.config.metrics.model_name
        checkpoints = list(ckpt_dir.glob("epoch*.pth"))
        if not checkpoints:
            raise FileNotFoundError(f"No checkpoint found in {ckpt_dir}")
        return max(checkpoints, key=lambda p: float(p.stem.split("_acc")[1]))
    
    def _save_artifacts(self, metrics: dict, classes: list[str]):
        import numpy as np
        cm = np.array(metrics["confusion_matrix"])

        save_confusion_matrix(cm, classes, self.config.output_dir / "confusion_matrix.png")

        with open(self.config.output_dir / "classification_report.json", "w") as f:
            json.dump(metrics["report"], f, indent=4)

        logger.info(f"Saved evaluation artifacts to {self.config.output_dir}")
    
    def _save_gradcam_samples(self, model, eval_loader, classes, n_samples_per_class: int = 1):
        gradcam = GradCAM(model)
        gradcam_dir = self.config.output_dir / "gradcam"
        gradcam_dir.mkdir(exist_ok=True)

        seen_classes = set()
        for imgs, labels in eval_loader:
            for img, label in zip(imgs, labels):
                label = label.item()
                if label in seen_classes:
                    continue
                seen_classes.add(label)

                cam = gradcam.generate(img, target_class=label)

                img_np = img.permute(1, 2, 0).cpu().numpy()
                img_np = (img_np * [0.229, 0.224, 0.225] + [0.485, 0.456, 0.406]) * 255
                img_np = img_np.clip(0, 255).astype(np.uint8)

                save_gradcam_overlay(img_np, cam, gradcam_dir / f"{classes[label]}.png")

                if len(seen_classes) >= len(classes):
                    return

In [14]:
try:
    cfg_manager = EvalConfigurationManager()
    eval_cfg    = cfg_manager.get_eval_config()
    evaluation  = Evaluation(config=eval_cfg)
    evaluation.evaluate()
except Exception as e:
    raise CustomException(e, sys)

[2026-06-18 16:49:25,850: INFO: __init__: yaml file: config/config.yaml loaded successfully]
[2026-06-18 16:49:25,855: INFO: __init__: yaml file: params.yaml loaded successfully]
[2026-06-18 16:49:25,858: INFO: __init__: created directory at: artifacts]
[2026-06-18 16:49:25,863: INFO: __init__: created directory at: artifacts/evaluation]
[2026-06-18 16:49:25,864: INFO: __init__: created directory at: artifacts/evaluation/efficientnet]
[2026-06-18 16:49:25,879: INFO: 3980762470: Evaluation dataset: 4,073 Classes: ['1. Eczema 1677', '10. Warts Molluscum and other Viral Infections - 2103', '2. Melanoma 15.75k', '3. Atopic Dermatitis - 1.25k', '4. Basal Cell Carcinoma (BCC) 3323', '5. Melanocytic Nevi (NV) - 7970', '6. Benign Keratosis-like Lesions (BKL) 2624', '7. Psoriasis pictures Lichen Planus and related diseases - 2k', '8. Seborrheic Keratoses and other Benign Tumors - 1.8k', '9. Tinea Ringworm Candidiasis and other Fungal Infections - 1.7k']]
[2026-06-18 16:49:33,185: INFO: 39807624